In [ ]:
import anthropic
import config
import logging
import re

logger = logging.getLogger(__name__)

In [2]:
_client = None

def get_client() -> anthropic.Anthropic:
    global _client
    if _client is None:
        _client = anthropic.Anthropic(api_key=config.ANTHROPIC_API_KEY)
    return _client


In [3]:
SYSTEM_PROMPT = """You are a helpful assistant that answers questions about Toronto
municipal by-laws and Ontario tenancy law.

Rules you must follow:
1. Answer ONLY using the provided context passages. Do not use outside knowledge.
2. Cite every factual claim with the section number it came from, like this: [§ 547-1.2]
3. If the context does not contain enough information to answer, say exactly:
   "I could not find a relevant passage in the by-laws to answer this question.
   Please consult the City of Toronto website or a legal professional."
4. Never speculate or infer beyond what the passages explicitly state.
5. If the answer may have changed due to recent amendments, note this explicitly."""

In [4]:
def format_context(chunks):
    parts = []
    for i, chunk in enumerate(chunks, 1):
        parts.append(
            f"[Passage {i}]\n"
            f"Section: § {chunk.section_id} — {chunk.section_title}\n"
            f"Domain: {chunk.domain}\n"
            f"Text: {chunk.text.strip()}\n"
        )
    return "\n---\n".join(parts)

In [5]:
from dataclasses import dataclass
@dataclass
class RetrievedChunk:
    chunk_id: str
    domain: str
    section_id: str
    section_title: str
    parent_section: str
    text: str
    page: int | None
    score: float


In [26]:
chunks = [RetrievedChunk(chunk_id='noise::591-2.3::0', domain='noise', section_id='591-2.3', section_title='Construction.', parent_section='ARTICLE 2', text='[Amended 2024-03-22 by By-law 268-2024] No person shall emit or cause or permit the emission of sound resulting from construction or any operation of construction equipment that is clearly audible: A. From 7 p.m. to 7 a.m. the next day, except until 9 a.m. on Saturdays; and/or B. All day on Sundays and statutory holidays.', page=6, score=0.03252247488101534),
 RetrievedChunk(chunk_id='noise::591-2.4::0', domain='noise', section_id='591-2.4', section_title='Loading and unloading.', parent_section='ARTICLE 2', text='[Amended 2019-07-18 by By-law 1102-2019; 2022-07-22 by By-law 1102-2022] A. No person shall emit or cause or permit the emission of sound resulting from loading, unloading, delivering, packing, unpacking, and otherwise handling any containers, products or materials from 11 p.m. to 7 a.m. the next day, except until 9 a.m. on Saturdays, Sundays and statutory holidays. B. In accordance with section 115.1 of the City of Toronto Act, 2006, Subsection A does not apply to the delivery of goods to the following, except as otherwise authorized by a regulation made under that section: (1) Retail business establishments. (2) Restaurants, including cafes and bars. (3) Hotels and motels. (4) Goods distribution facilities.', page=6, score=0.0315136476426799),
 RetrievedChunk(chunk_id='noise::591-2.6::0', domain='noise', section_id='591-2.6', section_title='Power devices.', parent_section='ARTICLE 2', text="[Amended 2022-07-22 by By-law 1102-20224] A. No person shall emit or cause or permit the emission of sound from a power device from 7 p.m. until 8 a.m. the next day, except until 9 a.m. on Saturdays, Sundays and statutory holidays. 3 Editor’s Note: By-law 288-2024 is deemed to have come into force on June 1, 2024. 4 Editor's Note: Section 1b of By-law 1102-2022 came into force on September 1, 2022. B. Subsection A does not apply to a power device used to maintain a golf course or public park or carry out City operations including services contracted by the City.", page=7, score=0.03149801587301587),
 RetrievedChunk(chunk_id='noise::591-3.2::4', domain='noise', section_id='591-3.2', section_title='Exemption permits.', parent_section='ARTICLE 3', text='F. Where an application for an exemption permit is made for continuous concrete pouring or large crane work, only Subsections A, B, B.1, C(2), (3) and (4), G and H apply and the Executive Director may issue the exemption permit subject to the conditions in Subsections D(1), (2), (7) and the conditions that: (1) The permission granted shall be for the date and times for each event or activity as set out in the exemption permit with overnight events or activities discouraged; (2) Notice for continuous concrete pouring and large crane work shall be distributed by the permit holder to those within a 120 metre radius of the activity at least 7 days prior to the start of such activity; and (3) The Executive Director shall provide a final copy of any exemption permit issued under this Subsection to the Councillor of any ward where such activity is to be conducted and, where the activity is to being conducted on a boundary street between wards, to the Councillors of the adjoining wards. F.1 In addition to those conditions set out in Subsection D, where the noise described in an exemption permit application is categorized as ‘Level 2’ or ‘Level 3’ under the Exemption Permit Screening Criteria, the Executive Director may impose the following conditions on the exemption permit: (1) The permit holder must distribute a notice of the exemption permit, in a form and manner satisfactory to the Executive Director, to those within a 120-metre radius of the activity at least 7 days prior to the start of the event or activity; (2) The permit holder must adhere to specific orientation of equipment for the duration of the event or activity, as determined by the Executive Director; (3) The permit holder must install sound dampeners or deadeners, or any other noise protection equipment determined by the Executive Director for the duration of the event or activity. F.2.', page=10, score=0.03149801587301587),
 RetrievedChunk(chunk_id='noise::591-3.2::5', domain='noise', section_id='591-3.2', section_title='Exemption permits.', parent_section='ARTICLE 3', text="F.2. In determining which additional conditions under Subsection F.1 are appropriate, the Executive Director will consider criteria, including but not limited to: (1) The duration of the event or activity and the hours the event or activity will be occurring; (2) The total number of participants or attendees at an event or activity with amplified sound or the type of construction development; (3) The proximity of the noise to a residential area and the likelihood that the noise for which an exemption is requested may negatively affect persons in that residential area; and (4) The applicant’s compliance with this chapter, including any previous exemption permits, if any, issued to them. G. Despite anything contained in § 591-3.2., where an application for an exemption permit is made by the City or any of its agencies, boards or commissions: (1) The application shall be submitted directly to the Executive Director by the City department, agency, board or commission seeking the exemption permit. (2) The fees in Chapter 441, Fees and Charges, do not apply. (3) Subsections C(3)(e) and (f) do not apply. G.1 Despite anything contained in § 591-3.2., where an application for an exemption permit is made by a not-for-profit corporation or a grassroots cultural organization, the applicant will not be required to pay the associated exemption permit application fee in Chapter 441, Fees and Charges. [Amended 2024-11-14 by By-law 1231-2024] H. The Executive Director may revoke an exemption permit, with or without notice, if there is non-compliance any of the exemption permit's conditions. I. The Executive Director will develop Exemption Permit Screening Criteria for the purposes of categorizing types of events or activities contained in an exemption permit application and use the Exemption Permit Screening Criteria in the assessment of exemption applications, including determining the applicable conditions and applicable fees.", page=10, score=0.01639344262295082)]

In [27]:
query = "What are the quiet hours for construction noise in Toronto?"

In [28]:
client = get_client()
context = format_context(chunks)

user_message = f"""Context passages:

    {context}

    Question: {query}

    Answer the question using only the context passages above. Cite section numbers."""

response = client.messages.create(
    model=config.GENERATION_MODEL,
    max_tokens=1000,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": user_message}],
)

answer = response.content[0].text.strip()

In [29]:
print(answer)

## Quiet Hours for Construction Noise in Toronto

Under Toronto's noise by-law, **no person shall emit or cause or permit the emission of sound resulting from construction** that is clearly audible during the following times [§ 591-2.3]:

- **7 p.m. to 7 a.m. the next day** (on weekdays)
- **7 p.m. to 9 a.m. the next day** (on Saturdays — construction is prohibited until 9 a.m., not just 7 a.m.)
- **All day on Sundays and statutory holidays** (construction noise is prohibited for the entire day)

In other words, construction is generally only permitted between **7 a.m. and 7 p.m. on weekdays**, and between **9 a.m. and 7 p.m. on Saturdays**. It is completely prohibited on Sundays and statutory holidays [§ 591-2.3].

> ⚠️ **Note:** This section was **amended on 2024-03-22 by By-law 268-2024**, so if you are dealing with a situation prior to that date, the rules may have differed. It is advisable to confirm the current version on the [City of Toronto website](https://www.toronto.ca) or c

In [30]:
sources = re.findall(r'§\w?\s*([\d]+-[\d.]+)', answer)

In [31]:
print("Cited sources:", set(sources))

Cited sources: {'591-2.3'}
